# BCSR持续学习实验 - Colab优化版

本notebook针对Google Colab T4 GPU环境优化BCSR性能。

## 优化要点

1. **预采样**：对>3000样本先预采样到2000-3000个
2. **减少迭代**：外层迭代从5次降到2次
3. **GPU优化**：batch_size=8192（T4 Tensor Core友好）
4. **混合精度**：使用FP16减少内存占用

## 使用方法

1. 运行下面的安装单元格
2. 运行测试单元格
3. 观察性能提升效果

In [ ]:
# 安装依赖
!pip install torch torchvision numpy scipy scikit-learn matplotlib seaborn tqdm pandas -q

# 克隆项目（如果需要）
# !git clone https://github.com/your-repo/core-set-benchmark.git
# %cd core-set-benchmark

import os
import sys
sys.path.append('/content/f--paper-code/coreset_benchmark')  # 调整路径

import torch
print(f"PyTorch版本: {torch.__version__}")
print(f"CUDA可用: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"显存: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## 快速测试BCSR性能

In [ ]:
# 测试BCSR在大数据集上的性能
import time
import torch
from src.coreset.bcsr_coreset import BCSRCoreset
from src.models.cnn import CNN_MNIST

# 设置设备
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"使用设备: {device}")

# 创建测试数据（模拟Task 0的数据）
torch.manual_seed(42)
n_samples = 12665  # MNIST Task 0的样本数
data = torch.randn(n_samples, 1, 28, 28).to(device)
labels = torch.randint(0, 2, (n_samples,)).to(device)  # 类别0和1

# 创建模型
model = CNN_MNIST(num_classes=2).to(device)

# 创建Colab优化的BCSR
print("\n创建BCSR选择器...")
bcsr = BCSRCoreset(
    learning_rate_inner=0.01,
    learning_rate_outer=3.0,  # Colab优化：降低到3
    num_inner_steps=1,
    num_outer_steps=2,  # Colab优化：降低到2
    beta=0.1,
    device=device
)

# 运行BCSR选择
print("\n开始BCSR选择...")
start_time = time.time()

selected_X, selected_y, info = bcsr.coreset_select(
    X=data,
    y=labels,
    coreset_size=1000,
    model=model
)

elapsed = time.time() - start_time
print(f"\n完成! 用时: {elapsed:.1f}秒")
print(f"选择了 {len(selected_X)} 个样本")
print(f"权重均值: {info['weights_mean']:.4f}")
print(f"\n类别分布:")
unique, counts = torch.unique(torch.from_numpy(selected_y), return_counts=True)
for label, count in zip(unique.tolist(), counts.tolist()):
    print(f"  类别 {label}: {count} 样本")

## 性能对比

| 优化 | 12665样本选1000 | 说明 |
|------|-----------------|------|
| **未优化** | 可能>10分钟 | 5次外层迭代，完整双层优化 |
| **预采样+2次迭代** | ~30-60秒 | 预采样到2000样本，2次外层迭代 |
| **uniform方法** | ~1秒 | 不使用模型，最快 |

## 完整实验示例

```python
# 在Colab中运行完整实验
from notebooks.colab_optimized import run_colab_optimized_experiment

# 运行优化后的实验
run_colab_optimized_experiment(
    dataset='MNIST',
    num_tasks=2,  # 先用2个任务测试
    num_classes_per_task=2,
    memory_size=2000,
    device='cuda'
)
```


## Colab使用技巧

### 1. 启用GPU
```
# 运行时 -> 更改运行时类型 -> GPU
```

### 2. 监控GPU使用
```python
import torch
print(f"GPU显存已用: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")
```

### 3. 如果仍然OOM
- 减少 `num_samples` 或 `memory_size`
- 使用uniform方法代替BCSR
- 减少batch_size